In [ ]:
import pandas as pd
import requests
import zipfile
import joblib
import os
from io import BytesIO

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import seaborn as sns

#### Download the UCI dataset

In [ ]:
# Official UCI dataset location
dataset_url = (
    "https://archive.ics.uci.edu/static/public/"
    "222/bank+marketing.zip"
)

try:

  # Download the dataset archive
  response = requests.get(dataset_url, timeout=60)
  response.raise_for_status()
except Exception as e:
  print(f"Error downloading the dataset: {e}")
  exit(1)

# Open the downloaded ZIP file
outer_archive = zipfile.ZipFile(BytesIO(response.content))


# Find the complete additional dataset
inner_zip_name = next(
    name for name in outer_archive.namelist()
    if "bank-additional" in name and name.endswith(".zip")
)

inner_zip_bytes = outer_archive.read(inner_zip_name)
inner_archive = zipfile.ZipFile(BytesIO(inner_zip_bytes))

# Find the actual CSV file inside the inner zip
csv_filename = next(
    name for name in inner_archive.namelist()
    if name.endswith("bank-additional") or name.endswith(".csv")
)

# # Read the CSV (the original file uses semicolon separators)
with inner_archive.open(csv_filename) as file:
    bank_data = pd.read_csv(file, sep=";")

bank_data.head(5)

# Initial dataset verification
print("Dataset loaded successfully!")
print("-" * 45)

print("Dataset shape:", bank_data.shape)
print("Number of records:", bank_data.shape[0])
print("Number of columns:", bank_data.shape[1])

print("\nColumn names:")
print(bank_data.columns.tolist())

print("\nTarget distribution:")
print(bank_data["y"].value_counts())

print("\nTarget distribution (%):")
print(
    bank_data["y"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

# Verify the assignment requirements
assert bank_data.shape[0] >= 500, "Insufficient records"
assert bank_data.shape[1] - 1 >= 12, "Insufficient features"
assert "y" in bank_data.columns, "Target column missing"

#### Structure of the bank marketing data


In [ ]:

print("Dataset information")
print("=" * 72)

bank_data.info()

print("\nFirst five observations")
print("=" * 72)

display(bank_data.head())

print("\nStatistical summary of numerical columns")
print("=" * 72)

display(bank_data.describe().T)

print("\nStatistical summary of categorical columns")
print("=" * 72)

display(bank_data.describe(include="object").T)


#### Identify feature types and missing values

In [ ]:
numerical_features = bank_data.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = bank_data.select_dtypes(
    include=["object"]
).columns.tolist()

# The target is not an input feature
numerical_features.remove("duration")  # Excluded from modelling
categorical_features.remove("y")

print("Numerical features:", numerical_features)
print("\nNumber of numerical features:", len(numerical_features))

print("\nCategorical features:", categorical_features)
print("\nNumber of categorical features:", len(categorical_features))

# Check missing values
print("\nMissing value count per column")
print("=" * 55)

missing_values = bank_data.isnull().sum()
display(missing_values[missing_values > 0])

print("\nTotal missing values:", missing_values.sum())

# Check unknown values used in the original dataset
print("\nUnknown values per column")
print("=" * 55)

unknown_counts = (
    bank_data.astype("object")
    .eq("unknown")
    .sum()
)

display(unknown_counts[unknown_counts > 0])


#### Visualize the target variable distribution

In [ ]:

target_counts = bank_data["y"].value_counts()

plt.figure(figsize=(7, 5))

sns.countplot(
    data=bank_data,
    x="y",
    hue="y",
    order=["no", "yes"],
    palette="Set2",
    legend=False
)

plt.title("Distribution of Bank Marketing Target Classes")
plt.xlabel("Term Deposit Subscription")
plt.ylabel("Number of Customers")

plt.tight_layout()
plt.show()


#### Prepare input features and target variables

In [ ]:

# Remove duration because it is unavailable before the marketing call
model_data = bank_data.drop(columns=["duration"]).copy()

# Separate input features and target
X = model_data.drop(columns=["y"])
y = model_data["y"].map({"no": 0, "yes": 1})

# Split the dataset while maintaining class proportions
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training and testing data prepared")
print("-" * 72)

print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])

print("\nTraining feature count:", X_train.shape[1])
print("Testing feature count:", X_test.shape[1])

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).round(4))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True).round(4))

print("\nTarget mapping: no = 0, yes = 1")


#### Define preprocessing operations

In [ ]:
# Scale numerical features and encode categorical features
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            StandardScaler(),
            numerical_features
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

print("Preprocessing configuration created.")
print("Numerical transformation: StandardScaler")
print("Categorical transformation: OneHotEncoder")

#### 1. Train the Logistic Regression classifier

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Combine preprocessing and classification
logistic_model = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

# Train using only the training dataset
logistic_model.fit(X_train, y_train)

# Generate predictions on unseen test records
logistic_predictions = logistic_model.predict(X_test)

# Obtain probability estimates for AUC calculation
logistic_probabilities = logistic_model.predict_proba(X_test)[:, 1]

print("Logistic Regression training completed.")
print("Number of test predictions:", len(logistic_predictions))
print("Model pipeline:", logistic_model)


#### Evaluate the model

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report
)

# A common evaluation function

def evaluate_classifier(actual, predicted, probabilities):
    results = {
        "Accuracy": accuracy_score(actual, predicted),
        "AUC": roc_auc_score(actual, probabilities),
        "Precision": precision_score(
            actual, predicted, zero_division=0
        ),
        "Recall": recall_score(
            actual, predicted, zero_division=0
        ),
        "F1 Score": f1_score(
            actual, predicted, zero_division=0
        ),
        "MCC": matthews_corrcoef(actual, predicted)
    }

    return results


# Evaluate Logistic Regression
logistic_results = evaluate_classifier(
    y_test,
    logistic_predictions,
    logistic_probabilities
)

print("Logistic Regression - Evaluation Results")
print("=" * 50)

for metric, value in logistic_results.items():
    print(f"{metric:15s}: {value:.4f}")

print("\nConfusion Matrix")
print(confusion_matrix(y_test, logistic_predictions))

print("\nClassification Report")
print(
    classification_report(
        y_test,
        logistic_predictions,
        target_names=["No", "Yes"],
        zero_division=0
    )
)


#### Train the Decision Tree Model

In [ ]:

from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline

decision_tree_model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("classifier", DecisionTreeClassifier(
        max_depth=10,
        random_state=42
    ))
])

# Train the model using the training data
decision_tree_model.fit(X_train, y_train)

# Generate predictions for the test data
decision_tree_predictions = decision_tree_model.predict(X_test)

# Get probabilities for the positive class
decision_tree_probabilities = (
    decision_tree_model.predict_proba(X_test)[:, 1]
)

print("Decision Tree training completed.")
print("Number of test predictions:", len(decision_tree_predictions))


#### Evaluate the decision tree model

In [ ]:

# Calculate evaluation metrics
decision_tree_results = evaluate_classifier(
    y_test,
    decision_tree_predictions,
    decision_tree_probabilities
)

print("\nDecision Tree - Evaluation Results")
print("=" * 50)

for metric, value in decision_tree_results.items():
    print(f"{metric:<15}: {value:.4f}")

print("\nConfusion Matrix")
print(confusion_matrix(y_test, decision_tree_predictions))

print("\nClassification Report")
print(classification_report(
    y_test,
    decision_tree_predictions,
    target_names=["No", "Yes"],
    zero_division=0
))


#### Train the K-Nearest Neighbour model

In [ ]:

from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline

knn_model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("classifier", KNeighborsClassifier(n_neighbors=5))
])

# Train the model
knn_model.fit(X_train, y_train)

# Predict the test-set classes
knn_predictions = knn_model.predict(X_test)

# Get probabilities for the positive class
knn_probabilities = knn_model.predict_proba(X_test)[:, 1]

print("KNN training completed.")
print("Number of test predictions:", len(knn_predictions))


#### Evaluate the KNN Model

In [ ]:

knn_results = evaluate_classifier(
    y_test,
    knn_predictions,
    knn_probabilities
)

print("\nKNN - Evaluation Results")
print("=" * 50)

for metric, value in knn_results.items():
    print(f"{metric:<15}: {value:.4f}")

print("\nConfusion Matrix")
print(confusion_matrix(y_test, knn_predictions))

print("\nClassification Report")
print(classification_report(
    y_test,
    knn_predictions,
    target_names=["No", "Yes"],
    zero_division=0
))


#### Dense preprocessor and train Gaussian Naive Bayes

In [ ]:

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import GaussianNB

# Create a separate preprocessor that returns a dense array
dense_preprocessor = ColumnTransformer(
    transformers=[
        ("numerical", StandardScaler(), numerical_features),
        ("categorical",
         OneHotEncoder(
             handle_unknown="ignore",
             sparse_output=False
         ),
         categorical_features)
    ]
)

# Create the Gaussian Naive Bayes pipeline
naive_bayes_model = Pipeline(steps=[
    ("preprocessing", dense_preprocessor),
    ("classifier", GaussianNB())
])

# Train the model
naive_bayes_model.fit(X_train, y_train)

# Predict test-set classes
naive_bayes_predictions = naive_bayes_model.predict(X_test)

# Obtain probabilities for the positive class
naive_bayes_probabilities = (
    naive_bayes_model.predict_proba(X_test)[:, 1]
)

print("Gaussian Naive Bayes training completed.")
print("Number of test predictions:", len(naive_bayes_predictions))


#### Evaluate Evaluate Gaussian Naive Bayes Model

In [ ]:

naive_bayes_results = evaluate_classifier(
    y_test,
    naive_bayes_predictions,
    naive_bayes_probabilities
)

print("\nGaussian Naive Bayes - Evaluation Results")
print("=" * 50)

for metric, value in naive_bayes_results.items():
    print(f"{metric:<15}: {value:.4f}")

print("\nConfusion Matrix")
print(confusion_matrix(y_test, naive_bayes_predictions))

print("\nClassification Report")
print(classification_report(
    y_test,
    naive_bayes_predictions,
    target_names=["No", "Yes"],
    zero_division=0
))


#### Train the Linear SVM Model

In [ ]:

from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline

svm_model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("classifier", LinearSVC(
        C=1.0,
        max_iter=5000,
        random_state=42
    ))
])

# Train the model
svm_model.fit(X_train, y_train)

# Predict classes for the test data
svm_predictions = svm_model.predict(X_test)

# Obtain decision scores for ROC AUC
svm_decision_scores = svm_model.decision_function(X_test)

print("Linear SVM training completed.")
print("Number of test predictions:", len(svm_predictions))


#### Evaluate the Linear SVM Model

In [ ]:

svm_results = evaluate_classifier(
    y_test,
    svm_predictions,
    svm_decision_scores
)

print("\nLinear SVM - Evaluation Results")
print("=" * 50)

for metric, value in svm_results.items():
    print(f"{metric:<15}: {value:.4f}")

print("\nConfusion Matrix")
print(confusion_matrix(y_test, svm_predictions))

print("\nClassification Report")
print(classification_report(
    y_test,
    svm_predictions,
    target_names=["No", "Yes"],
    zero_division=0
))


#### Consolidate results from all five classifiers

In [ ]:

model_results = {
    "Logistic Regression": logistic_results,
    "Decision Tree": decision_tree_results,
    "KNN": knn_results,
    "Gaussian Naive Bayes": naive_bayes_results,
    "Linear SVM": svm_results
}

# Convert the results into a comparison table
comparison_df = pd.DataFrame(model_results).T

# Display results rounded to four decimal places
display(comparison_df.round(4))

# Save the comparison table
comparison_df.to_csv("model_comparison.csv", index=True)

print("Model comparison table saved successfully.")


#### Save the five trained models

In [ ]:
os.makedirs("models", exist_ok=True)

# Storing all five complete pipelines
trained_models = {
    "Logistic Regression": logistic_model,
    "Decision Tree": decision_tree_model,
    "KNN": knn_model,
    "Gaussian Naive Bayes": naive_bayes_model,
    "Linear SVM": svm_model
}

# Save models and their preprocessing pipelines
joblib.dump(trained_models, "models/trained_models.joblib")

print("All five trained models saved successfully.")

# Verify the saved file
print("Saved file exists:", os.path.exists("models/trained_models.joblib"))


#### Prepare test data

In [ ]:

# Prepare the held-out test dataset
test_data = X_test.copy()

# Convert target labels back to their original form
test_data["y"] = y_test.map({
    0: "no",
    1: "yes"
})

# Export the test dataset
test_data.to_csv("test_data.csv", index=False)

print("Test dataset exported successfully.")
print("Test dataset shape:", test_data.shape)

display(test_data.head())


In [ ]:

# Load the saved model dictionary
loaded_models = joblib.load("models/trained_models.joblib")

# Verify each saved model by generating predictions
for model_name, model in loaded_models.items():
    predictions = model.predict(X_test)

    print(
        f"{model_name}: "
        f"{len(predictions)} predictions generated"
    )


In [ ]:
!pip install streamlit

In [ ]:
import os
import joblib
import sklearn
import streamlit
import pandas as pd
import numpy as np

print("Python packages:")
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Streamlit:", streamlit.__version__)

print("\nModel artifact:")
model_path = "models/trained_models.joblib"
print("Exists:", os.path.exists(model_path))

if os.path.exists(model_path):
    print("Size (MB):", round(os.path.getsize(model_path) / (1024 * 1024), 2))
    saved_models = joblib.load(model_path)
    print("Models:", list(saved_models.keys()))

print("\nTest data:")
print("Shape:", pd.read_csv("test_data.csv").shape)

In [ ]:
%%writefile app.py

import os
import joblib
import numpy as np
import pandas as pd
import streamlit as st
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

# --------------------------------------------------
# 1. Basic application settings
# --------------------------------------------------

st.set_page_config(
    page_title="Bank Marketing Classifier",
    page_icon="📊",
    layout="wide"
)

st.title("Bank Marketing Classification")
st.write(
    "This application evaluates pre-trained machine learning "
    "classifiers on uploaded bank marketing test data."
)

st.info(
    "Upload a labelled CSV file containing the model input features "
    "and the target column 'y'. The application uses saved models "
    "and does not perform any model training."
)

# --------------------------------------------------
# 2. Load the saved machine learning models
# --------------------------------------------------

MODEL_PATH = Path("model/trained_models.joblib")


@st.cache_resource
def load_saved_models():
    """Load the five trained pipelines from the model artifact."""
    if not MODEL_PATH.exists():
        raise FileNotFoundError(
            f"Could not find the model artifact: {MODEL_PATH}"
        )

    return joblib.load(MODEL_PATH)


try:
    trained_models = load_saved_models()
except Exception as error:
    st.error(f"Unable to load the saved models: {error}")
    st.stop()


# --------------------------------------------------
# 3. Identify the required input features
# --------------------------------------------------

# The feature names are taken from the fitted pipeline.
required_features = list(
    trained_models["Logistic Regression"].feature_names_in_
)

st.sidebar.header("Model Selection")

selected_model_name = st.sidebar.selectbox(
    "Choose a classification model",
    list(trained_models.keys())
)

st.sidebar.caption(
    "All selected models are loaded from the saved artifact."
)


# --------------------------------------------------
# 4. Upload the labelled test dataset
# --------------------------------------------------

st.subheader("Upload Test Dataset")

uploaded_file = st.file_uploader(
    "Choose a CSV file",
    type=["csv"],
    help="The CSV must contain all 19 input features and the target column y."
)

if uploaded_file is None:
    st.warning("Please upload a labelled CSV file to begin evaluation.")
    st.stop()


# --------------------------------------------------
# 5. Read and validate the uploaded data
# --------------------------------------------------

try:
    uploaded_data = pd.read_csv(uploaded_file)
except Exception as error:
    st.error(f"Unable to read the uploaded CSV file: {error}")
    st.stop()


# Remove accidental index columns, if present.
uploaded_data = uploaded_data.loc[
    :, ~uploaded_data.columns.str.startswith("Unnamed:")
]

if "y" not in uploaded_data.columns:
    st.error("The uploaded CSV must contain a target column named 'y'.")
    st.stop()

missing_features = [
    feature for feature in required_features
    if feature not in uploaded_data.columns
]

if missing_features:
    st.error(
        "The uploaded file is missing these required features: "
        + ", ".join(missing_features)
    )
    st.stop()

if uploaded_data.empty:
    st.error("The uploaded CSV does not contain any records.")
    st.stop()

# Keep the original feature order used during model training.
X_uploaded = uploaded_data[required_features].copy()

# Convert the target labels into the numerical format used in training.
target_mapping = {
    "no": 0,
    "yes": 1,
    "0": 0,
    "1": 1
}

y_uploaded = (
    uploaded_data["y"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map(target_mapping)
)

if y_uploaded.isna().any():
    st.error(
        "The target column contains unsupported values. "
        "Expected values are 'yes' and 'no' (or 1 and 0)."
    )
    st.stop()

y_uploaded = y_uploaded.astype(int)

if y_uploaded.nunique() < 2:
    st.error(
        "AUC cannot be evaluated because the uploaded data "
        "contains only one target class. Please upload data "
        "containing both 'yes' and 'no' records."
    )
    st.stop()


# --------------------------------------------------
# 6. Generate predictions using the selected model
# --------------------------------------------------

selected_model = trained_models[selected_model_name]

try:
    predictions = selected_model.predict(X_uploaded)

    # Logistic Regression, Decision Tree, KNN and Gaussian NB
    # provide probabilities. Linear SVM provides decision scores.
    if hasattr(selected_model, "predict_proba"):
        prediction_scores = selected_model.predict_proba(
            X_uploaded
        )[:, 1]
    else:
        prediction_scores = selected_model.decision_function(
            X_uploaded
        )

except Exception as error:
    st.error(f"Prediction failed: {error}")
    st.stop()


# --------------------------------------------------
# 7. Calculate model evaluation metrics
# --------------------------------------------------

accuracy = accuracy_score(y_uploaded, predictions)

auc = roc_auc_score(
    y_uploaded,
    prediction_scores
)

precision = precision_score(
    y_uploaded,
    predictions,
    zero_division=0
)

recall = recall_score(
    y_uploaded,
    predictions,
    zero_division=0
)

f1 = f1_score(
    y_uploaded,
    predictions,
    zero_division=0
)

mcc = matthews_corrcoef(
    y_uploaded,
    predictions
)


# --------------------------------------------------
# 8. Display the evaluation results
# --------------------------------------------------

st.divider()

st.subheader("Evaluation Results")

st.write(f"**Selected model:** {selected_model_name}")
st.write(f"**Number of evaluated records:** {len(X_uploaded):,}")

metric_columns = st.columns(3)

metric_values = [
    ("Accuracy", accuracy),
    ("AUC", auc),
    ("Precision", precision),
    ("Recall", recall),
    ("F1 Score", f1),
    ("MCC", mcc)
]

for index, (metric_name, metric_value) in enumerate(metric_values):
    with metric_columns[index % 3]:
        st.metric(
            label=metric_name,
            value=f"{metric_value:.4f}"
        )


# --------------------------------------------------
# 9. Display the confusion matrix
# --------------------------------------------------

st.divider()
st.subheader("Confusion Matrix")

matrix = confusion_matrix(
    y_uploaded,
    predictions,
    labels=[0, 1]
)

fig, ax = plt.subplots(figsize=(4, 3), dpi=100)

display = ConfusionMatrixDisplay(
    confusion_matrix=matrix,
    display_labels=["No", "Yes"]
)

display.plot(
    ax=ax,
    cmap="Blues",
    values_format="d",
    colorbar=False
)

ax.set_title(f"Confusion Matrix - {selected_model_name}")
ax.set_xlabel("Predicted Label")
ax.set_ylabel("Actual Label")

plt.tight_layout()
st.pyplot(fig)
plt.close(fig)

# --------------------------------------------------
# 10. Display the classification report
# --------------------------------------------------

st.divider()
st.subheader("Classification Report")

report = classification_report(
    y_uploaded,
    predictions,
    labels=[0, 1],
    target_names=["No", "Yes"],
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).transpose()

st.dataframe(
    report_df.round(4),
    use_container_width=True
)


# --------------------------------------------------
# 11. Show a preview of the uploaded data
# --------------------------------------------------

with st.expander("View uploaded dataset"):
    st.dataframe(
        uploaded_data.head(10),
        use_container_width=True
    )

st.success(
    "Evaluation completed successfully using the saved model. "
    "No model retraining was performed."
)

In [ ]:
!pip install -q pyngrok

In [ ]:
from google.colab import userdata
from pyngrok import ngrok
import subprocess
import time

# Read the token securely from Colab Secrets.
ngrok_token = userdata.get("NGROK_AUTH_TOKEN")
# ngrok.set_auth_token(ngrok_token)

# Start the Streamlit application.
process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(5)

# Create a public preview URL.
public_url = ngrok.connect(8501)

print("Streamlit application URL:", public_url)

In [ ]:
from pyngrok import ngrok

# Close existing tunnels.
try:
    ngrok.disconnect_all()
except Exception:
    pass

# Stop the existing ngrok agent.
try:
    ngrok.kill()
except Exception:
    pass

# Stop the previous Streamlit process, if it exists.
try:
    if process.poll() is None:
        process.terminate()
        process.wait(timeout=5)
except (NameError, Exception):
    pass

print("Previous processes cleaned up.")

In [ ]:
from pyngrok import ngrok

public_url = ngrok.connect(8501, "http")

print("Your Streamlit preview URL:")
print(public_url)

In [ ]:
from importlib.metadata import version

required_packages = [
    "numpy",
    "pandas",
    "scikit-learn",
    "matplotlib",
    "joblib",
    "streamlit"
]

with open("requirements.txt", "w") as file:
    for package in required_packages:
        package_version = version(package)
        file.write(f"{package}=={package_version}\n")

print("Updated requirements.txt:\n")

with open("requirements.txt", "r") as file:
    print(file.read())